In [4]:
FAILURE = "FAILURE"

def is_variable(expr):
    return isinstance(expr, str) and len(expr) > 0 and expr[0].isupper()

def is_constant(expr):
    return isinstance(expr, str) and len(expr) > 0 and expr[0].islower()

def occurs_check(var, expr):
    if var == expr:
        return True
    if isinstance(expr, (list, tuple)):
        return any(occurs_check(var, arg) for arg in expr[1:])
    return False

def apply_substitution(subst, expr):
    if subst is None or subst == FAILURE:
        return expr
    if is_variable(expr):
        return subst.get(expr, expr)
    if isinstance(expr, (list, tuple)):
        return [expr[0]] + [apply_substitution(subst, arg) for arg in expr[1:]]
    return expr

def compose_substitutions(s, subst):
    if subst is None:
        return s
    if s is None:
        return subst
    new_subst = {k: apply_substitution(s, v) for k, v in subst.items()}
    for k, v in s.items():
        if k not in new_subst:
            new_subst[k] = v
    return new_subst

def unify(psi1, psi2):
    if is_variable(psi1) or is_constant(psi1) or is_variable(psi2) or is_constant(psi2):
        if psi1 == psi2:
            return {}
        elif is_variable(psi1):
            if occurs_check(psi1, psi2):
                return FAILURE
            else:
                return {psi1: psi2}
        elif is_variable(psi2):
            if occurs_check(psi2, psi1):
                return FAILURE
            return {psi2: psi1}
        else:
            return FAILURE

    if not isinstance(psi1, (list, tuple)) or not isinstance(psi2, (list, tuple)):
        return FAILURE

    if psi1[0] != psi2[0]:
        return FAILURE

    if len(psi1) != len(psi2):
        return FAILURE

    subst = {}
    num_args = len(psi1) - 1
    for i in range(1, num_args + 1):
        arg1 = apply_substitution(subst, psi1[i])
        arg2 = apply_substitution(subst, psi2[i])
        s = unify(arg1, arg2)
        if s == FAILURE:
            return FAILURE
        subst = compose_substitutions(s, subst)
    return subst

if __name__ == "__main__":
    # Test 1: Standard unification
    L1 = ["P", "X", "a"]
    L2 = ["P", "b", "Y"]
    print("Test 1 Result:", unify(L1, L2))

    # Test 2: Unifying variables
    L3 = ["f", "X"]
    L4 = ["f", "b"]
    print("Test 2 Result:", unify(L3, L4))

    # Test 3: Unification failure due to arity mismatch
    L5 = ["P", "x"]
    L6 = ["P", "f", "X"]
    print("Test 3 Result:", unify(L5, L6))

Test 1 Result: {'X': 'b', 'Y': 'a'}
Test 2 Result: {'X': 'b'}
Test 3 Result: FAILURE
